# Import the required libraries

In [62]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pickle

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Import the dataset

In [64]:
dataset = pd.read_csv("ai_job_market_trend.csv")

In [65]:
dataset.head()

,job_id,job_role,experience_level,years_of_experience,education_level,skills,tools_and_technologies,certifications,industry,employment_type,location,company_size,annual_salary_usd,remote_friendly
0,1,AI Researcher,Entry,1,Bachelor's,"Python, Algorithm Design, Experimentation, Dee...",Overleaf,NaN,Robotics,Full-time,"Austin, TX",Large,90697,Hybrid
1,2,AI Researcher,Mid,3,Master's,"Mathematics, Statistics, Python, Publications,...","SLURM, PyTorch, Weights & Biases",ICLR Paper Publication,Tech,Full-time,"Chicago, IL",Large,142298,Yes
2,3,Quant Researcher,Mid,6,Master's,"Financial Modeling, R, Python, Statistics, Sto...","R, Python, KDB+",CFA Level I,Banking,Remote,"London, UK",Large,139116,Yes
3,4,AI Researcher,Mid,2,PhD,"Algorithm Design, Mathematics, Statistics, Res...","JAX, Overleaf",ICLR Paper Publication,Finance,Full-time,"Atlanta, GA",Startup,144867,No
4,5,ML Engineer,Mid,5,Bachelor's,"Model Deployment, APIs, SQL, Machine Learning,...","FastAPI, PyTorch",Azure AI Engineer,Finance,Full-time,"Dublin, Ireland",Mid-size,158956,Hybrid


In [66]:
dataset.shape

(5000, 14)

In [67]:
dataset.isnull().sum()

job_id                       0
job_role                     0
experience_level             0
years_of_experience          0
education_level              0
skills                       0
tools_and_technologies       0
certifications            1592
industry                     0
employment_type              0
location                     0
company_size                 0
annual_salary_usd            0
remote_friendly              0
dtype: int64

In [68]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   job_id                  5000 non-null   int64 
 1   job_role                5000 non-null   object
 2   experience_level        5000 non-null   object
 3   years_of_experience     5000 non-null   int64 
 4   education_level         5000 non-null   object
 5   skills                  5000 non-null   object
 6   tools_and_technologies  5000 non-null   object
 7   certifications          3408 non-null   object
 8   industry                5000 non-null   object
 9   employment_type         5000 non-null   object
 10  location                5000 non-null   object
 11  company_size            5000 non-null   object
 12  annual_salary_usd       5000 non-null   int64 
 13  remote_friendly         5000 non-null   object
dtypes: int64(3), object(11)
memory usage: 547.0+ KB


In [69]:
dataset.describe

<bound method NDFrame.describe of       job_id          job_role experience_level  years_of_experience  \
0          1     AI Researcher            Entry                    1   
1          2     AI Researcher              Mid                    3   
2          3  Quant Researcher              Mid                    6   
3          4     AI Researcher              Mid                    2   
4          5       ML Engineer              Mid                    5   
...      ...               ...              ...                  ...   
4995    4996  Quant Researcher              Mid                    6   
4996    4997    Data Scientist           Senior                   10   
4997    4998  Quant Researcher            Entry                    0   
4998    4999      Data Analyst              Mid                    5   
4999    5000  Quant Researcher           Senior                   13   

     education_level                                             skills  \
0         Bachelor's  Pyth

In [70]:
dataset.value_counts("industry")

industry
Healthcare       793
Finance          781
Tech             621
E-commerce       564
Defense          285
Retail           269
Insurance        212
Automotive       205
Education        194
SaaS             105
Legal             99
Hedge Fund        97
Robotics          96
Marketing         95
Media             91
Manufacturing     90
Banking           90
Logistics         85
Consulting        82
Telecom           76
Academia          70
Name: count, dtype: int64

In [71]:
dataset.value_counts("job_role")

job_role
Data Scientist              668
ML Engineer                 628
Data Analyst                604
NLP Engineer                568
AI Product Manager          539
Computer Vision Engineer    524
AI Researcher               509
Data Engineer               507
Quant Researcher            453
Name: count, dtype: int64

In [72]:
dataset.duplicated().sum()

0

# Data Preprocessing

In [74]:
dataset['certifications'] = dataset['certifications'].fillna('none')

In [75]:
dataset['skills_combined'] = (
    dataset['skills'].fillna('') + ' ' +
    dataset['tools_and_technologies'].fillna('') + ' ' +
    dataset['certifications'].fillna('')
)

In [76]:
count_vectorizer = CountVectorizer(max_features=80, ngram_range=(1, 1))
skills_matrix = count_vectorizer.fit_transform(dataset['skills_combined'])

In [77]:
categorical_cols = ['experience_level', 'education_level', 'industry',
                    'employment_type', 'company_size', 'remote_friendly']
numeric_cols     = ['years_of_experience', 'annual_salary_usd']

In [78]:
dataset_struct = dataset[categorical_cols + numeric_cols].copy()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    dataset_struct[col] = le.fit_transform(dataset_struct[col])
    label_encoders[col]=le

In [79]:
target_le = LabelEncoder()
y = target_le.fit_transform(dataset['job_role'])

In [80]:
X = sp.hstack([sp.csr_matrix(dataset_struct.values), skills_matrix])

In [81]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [101]:
print(X.shape, X_train.shape, X_test.shape)

(5000, 88) (4000, 88) (1000, 88)


# Model training and evaluation